In [7]:
import os
os.environ['https_proxy'] = '127.0.0.1:7897'

In [2]:
import torch
from transformers import AutoProcessor
from datasets import load_dataset


model_id ="HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
# SmolVLMProcessor
processor = AutoProcessor.from_pretrained(
    model_id
)

/home/escommune/miniforge3/envs/lerobot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from transformers import SmolVLMProcessor

# 手动实例化，不经过Auto自动分发，调试更清晰
smol_processor = SmolVLMProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

In [9]:
ds = load_dataset('merve/vqav2-small', split="train[:100]")

Generating validation split: 100%|██████████| 21435/21435 [00:34<00:00, 614.54 examples/s] 


ValueError: Unknown split "train". Should be one of ['validation'].

In [4]:
messages = [
      {
          "role": "user",
          "content": [
              {"type": "text", "text": "Answer briefly."},
              {"type": "image"},
              {"type": "text", "text": "whats your name"}
          ]
      },
      {
          "role": "assistant",
          "content": [
              {"type": "text", "text": "smolvlm-2"}
          ]
      }
  ]
text = processor.apply_chat_template(messages, add_generation_prompt=False)


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


In [6]:
processor.tokenizer.pad_token_id

2

In [5]:
image_token_id = 49190

def collate_fn(examples):
  texts = []
  images = []
  for example in examples:
      image = example["image"]
      if image.mode != 'RGB':
        image = image.convert('RGB')
      question = example["question"]
      answer = example["multiple_choice_answer"]
      messages = [
          {
              "role": "user",
              "content": [
                  {"type": "text", "text": "Answer briefly."},
                  {"type": "image"},
                  {"type": "text", "text": question}
              ]
          },
          {
              "role": "assistant",
              "content": [
                  {"type": "text", "text": answer}
              ]
          }
      ]
      text = processor.apply_chat_template(messages, add_generation_prompt=False)
      texts.append(text.strip())
      images.append([image])

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    batch["labels"] = labels

    return batch